# DAIS 2026 — Deployment & Setup

Short checklist to get a workspace ready for the DAIS 2026 runbooks
(`Genie.ipynb`, `AgentBricks.ipynb`, `LakebaseApps.ipynb`, `MLflow.ipynb`).
All four run against a single `all`-target deployment.

## 1. Workspace prerequisites

The `all` target touches every product. Confirm with your account team
**before** deploying — missing entitlements fail mid-run and waste a slot.

- **Region & cloud**: AWS or Azure workspace, region with Lakebase Autoscaling
  GA (e.g. `us-west-2`, `us-east-1`, `eu-west-1`). Free Edition will *not* work
  for `all` — use the `free` target instead.
- **Unity Catalog**: metastore attached; current user can `CREATE CATALOG`
  (or the target catalog already exists and you have `ALL PRIVILEGES`).
- **Serverless compute**: enabled for jobs, notebooks, SQL warehouses, and
  Lakeflow pipelines (Admin Console → Compute → Serverless).
- **Model Serving / Foundation Model APIs**: `databricks-claude-sonnet-4-5`
  available pay-per-token (default `LLM_MODEL`; override with
  `--params "LLM_MODEL=…"` if your workspace only has Llama).
- **Agent Bricks** (account-level preview, request via field team):
  - Knowledge Assistants v2.1 API
  - Genie Spaces API
  - Multi-Agent Supervisor API
- **Databricks Apps**: enabled (Admin Console → Previews → Databricks Apps).
- **Lakebase Autoscaling**: enabled and quota for ≥1 project / 3 logical
  databases (Admin Console → Previews → Lakebase).
- **AI SQL functions**: `ai_parse_document`, `ai_classify`, `ai_extract`,
  `ai_summarize` available in DBSQL (default in supported regions).
- **ABAC** (governance beat): `CREATE POLICY` requires DBR 16.4+ on the SQL
  warehouse readers use; the `all`-target warehouses default to current
  channel so this is normally a no-op check. The deployer needs `MANAGE` on
  the demo catalog. The ABAC tags applied in `Environment_Helpers` are
  plain UC tags (not governed tag policies) so no tag-policy admin
  privilege is required.
- **Data Quality Monitoring** (governance beat): Admin Console → Previews →
  *"Data quality monitoring with anomaly detection (workspace level)"* must
  be ON. If it isn't, the monitor cell in `Environment_Helpers` prints a
  loud `create_monitor failed: ...` error pointing back at the preview
  flag, and the rest of the stage continues. SDK is upgraded in-cell via
  `%pip install --upgrade databricks-sdk`, so the compute's baseline SDK
  version doesn't matter. The first scan after creation will not show
  meaningful health indicators — anomaly detection needs history before
  it can flag deviations.
- **Databricks One / Genie app**: presenter user has the **Databricks One**
  entitlement so `/one` loads with the Genie surface. iOS/Android Genie app
  installed and signed into the same workspace for the mobile beat.

## 2. Local prerequisites

- `databricks` CLI ≥ 0.275 (`databricks -v`)
- `jq` on `PATH` (used by the `cleanup` script)
- Authenticated to the target workspace:

  ```bash
  databricks auth login --host https://<workspace>.cloud.databricks.com
  ```

## 3. Deploy

```bash
databricks bundle deploy -t all
databricks bundle run    caspers -t all --params "CATALOG=<your_catalog>"
```

The job runs ~30–45 min end-to-end. Watch the run in the Jobs UI; every
task creates resources that the runbooks reference. `Evaluation` is the
final task — when it goes green, you're demo-ready.

> **Cache bug:** if files don't appear synced after a redeploy, `rm -rf
> .databricks .bundle` and redeploy. See `AGENTS.md` for the full workflow.

## 4. Discover Domains (Beta, one-time UI setup) WIP 

Discover Domains is Beta and has **no public REST/SDK/Terraform API yet**, so
domain creation itself is manual. Skip if your workspace doesn't have the
Discover preview flag enabled — the rest of the demo still works.

Domains are **tag-driven**: assets surface in a Domain because they carry the
matching governed tag, not because anyone pinned them. The pipelines create
three boolean governed tags and apply them to every tagged asset:

- `Environment_Helpers` creates the governed tag policies, tags every UC
  securable (catalogs / schemas / tables), and tags the AI/BI dashboards.
- `Genie_Spaces`, `Databricks_App_Refund_Manager`, and `Operational_App`
  each tag the workspace asset they create (Genie spaces, Apps) via
  `w.workspace_entity_tag_assignments`.

Once `bundle run caspers -t all` has finished, the three tags below already
exist and are bound to the right assets across UC + Genie + dashboards + apps.

In **Catalog → Discover → Domains → New domain**, create three domains and
select the matching existing governed tag for each:

| Domain | Governed tag (select existing) |
|---|---|
| **Operations** | `caspers_domain_operations = true` |
| **Revenue & Customers** | `caspers_domain_revenue = true` |
| **Compliance & Safety** | `caspers_domain_compliance = true` |

Assets can carry multiple domain tags — e.g. `lakeflow.silver_order_items`
is tagged with both `caspers_domain_operations` and `caspers_domain_revenue`,
so the same table shows in both Domains without duplication.

**Knowledge Assistants** cannot be added to a Domain today (no governed-tag
surface on KAs yet), so KA endpoints will be missing from the Domain views
even though the pipeline creates them. Everything else (UC tables, Genie
spaces, dashboards, apps) is covered.

**Tag-key naming note**: keys use underscores (`caspers_domain_operations`)
instead of dots. Both surfaces — UC table/column tags (`ALTER TABLE … SET
TAGS`) and the workspace-entity tag-assignment API — reject the characters
`, . : / - = % & ? > <` and leading/trailing spaces in tag keys
(`INVALID_PARAMETER_VALUE: Tag key contains reserved characters`).
Underscores are the safe shape that works for both surfaces, which is what
lets a single Domain in Discover surface UC tables AND Genie/apps/dashboards
side by side.

Domains survive `bundle destroy` and are independent of `_internal_state` —
delete them manually from the Discover UI if you change catalogs.

## 5. ABAC governance + data quality (governance demo beat)

`Environment_Helpers` does four `CREATE OR REPLACE POLICY` statements and
one schema-level `data_quality.create_monitor` call after Lakeflow finishes
materialising silver/gold. All four policies live in `${CATALOG}._security`
and are pure tag-driven (no hard-coded table lists), so they survive future
schema additions:

| # | Policy | Securable | UDF | Bypass group(s) |
|---|---|---|---|---|
| 1 | `caspers_mask_pii` (column mask) | `lakeflow` schema | `mask_pii` | `caspers_pii_readers` |
| 2 | `caspers_region_filter` (row filter) | `simulator` schema | `filter_by_region` | `caspers_geo_admins` (+ `caspers_us_users` / `caspers_emea_users` for the matching half) |
| 3 | `caspers_high_value_gate` (row filter) | `lakeflow` schema | `filter_high_value` | `caspers_finance`, `caspers_managers` |
| 4 | `caspers_regulated_docs` (row filter) | `food_safety` schema | `filter_regulated_docs` | `caspers_compliance` |

**Bypass shape.** Bypass logic lives **only** in the UDFs, which check
group membership via `is_account_group_member(...)`. The policies
themselves do **not** carry `EXCEPT <group>` clauses. This matters because
`CREATE POLICY` validates `EXCEPT` principals at creation time
(`PRINCIPAL_DOES_NOT_EXIST`), so any earlier shape that referenced
`caspers_pii_readers` etc. in the DDL hard-failed against any workspace
whose account didn't already have those seven groups.
`is_account_group_member` returns false for missing groups instead of
erroring, so the current shape creates cleanly with or without the demo
groups.

**Demo groups** (optional, only needed if you want to demo the bypass
perspective). The four UDFs check membership in these seven account-level
groups. Without them, the policies still apply — every non-admin user
sees masked/filtered data; workspace admins still bypass everything per
UC's built-in admin semantics. Create them in **Account console → User
management → Groups** if you want a non-admin demo user that sees the
unmasked perspective:

```
caspers_pii_readers   caspers_finance      caspers_compliance
caspers_us_users      caspers_managers
caspers_emea_users    caspers_geo_admins
```

**Data quality monitor.** Schema-level anomaly detection on
`${CATALOG}.food_safety` via `w.data_quality.create_monitor`. After the
helper task runs, results surface in Catalog Explorer → `food_safety` →
Quality tab. If the workspace preview flag is off the cell prints a loud
`create_monitor failed:` error pointing back at the preview toggle in
**Admin Console → Previews → "Data quality monitoring with anomaly
detection (workspace level)"** — the stage continues so the rest of the
governance beat still lands. Anomaly detection needs history to flag
regressions, so the Quality tab won't show meaningful indicators on the
first scan.

## 6. Pre-show warm-up (5 min before going on stage)

1. Open each runbook in the workspace and run its **pre-flight** cell with
   your `CATALOG` widget set — this prints fresh URLs and confirms every
   resource exists.
2. Click each app URL once (Ops Dashboard, Refund Manager) to dodge
   cold-start.
3. Click one sample question in each Genie space to warm
   `<catalog>-ops-warehouse` and `<catalog>-genie-warehouse`.
4. Open one Knowledge Assistant endpoint and the Supervisor endpoint to
   warm Model Serving.

## 7. Cleanup

```bash
databricks bundle run     cleanup -t all --var catalog=<your_catalog>
databricks bundle destroy -t all
```

`cleanup` is a **script**, not a job task — use `--var catalog=…`,
**not** `--params`. `destroy` only removes bundle-owned resources;
`cleanup` removes everything tracked in `<catalog>._internal_state.resources`
(Lakebase projects, apps, model endpoints, KAs, Genie spaces, …).